## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH06/ch06_SAM_audio.ipynb)                                             


#About the Notebook

This notebook demonstrates how to use [Meta’s SAM Audio model](https://arxiv.org/pdf/2512.18099) for advanced audio source separation. It walks you through model loading and inference to isolate specific sounds from a single audio file. The example focuses on separating **gunshots** and **a male voice** from the same recording.

Instead of extracting sources by predefined labels, the model isolates sound sources based on text prompts like *gunshots* or *male voice*. This makes audio separation flexible, intuitive, and fully prompt driven.

The result is a clean pair of separated tracks for analysis, post processing, or downstream machine learning tasks.



In [1]:
!git clone https://github.com/facebookresearch/sam-audio.git

Cloning into 'sam-audio'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 167 (delta 51), reused 35 (delta 35), pack-reused 93 (from 2)
Receiving objects: 100% (167/167), 15.70 MiB | 17.33 MiB/s, done.
Resolving deltas: 100% (80/80), done.


In [2]:
%cd sam-audio

/content/sam-audio


In [3]:
%%capture
!pip install .

In [4]:
%%capture
!pip install protobuf==3.20 gdown

# Imports

In [6]:
import sys
import torch
import torchaudio
from IPython.display import HTML, display
from IPython.display import Audio, display
from sam_audio import SAMAudio, SAMAudioProcessor

# Restart Session for Installed Packages

In [9]:
display(HTML("""
<div style="
    padding: 25px;
    background: #FFF1C4;
    border: 4px solid #D9534F;
    border-radius: 12px;
    text-align: center;
    font-family: Arial, sans-serif;
    box-shadow: 0 0 15px rgba(0,0,0,0.2);
">
  <h1 style="font-size: 2.2rem; margin-bottom: 0.5rem;">⚠️ SESSION RESTART REQUIRED ⚠️</h1>
  <p style="font-size: 1.25rem; font-weight: bold;">
    New packages were just installed and won't work until you restart the session.
  </p>
  <p style="font-size: 1.1rem; margin-top: 1rem;">
    👉 Go to: <b>Runtime → Restart session</b><br>
    🚫 This is not a runtime factory reset — just a session restart.
  </p>
</div>
"""))

sys.exit("⛔ Execution stopped: Please restart your session now.")


SystemExit: ⛔ Execution stopped: Please restart your session now.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Load model and processor

# ATT: You need to [request access for the SAM-audio model](https://huggingface.co/facebook/sam-audio-large)

In [1]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# sam-audio-small, sam-audio-base, and sam-audio-large are available
model = SAMAudio.from_pretrained("facebook/sam-audio-large").to(device).eval()
processor = SAMAudioProcessor.from_pretrained("facebook/sam-audio-large")



/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


config.json:   0%|          | 0.00/2.25k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE:   0%|          | 0.00/7.35k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.12G [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

# Download Audio File and Play it

In [2]:
# Download the file from Google Drive using its file ID
file_id = "1IymJaN-zOk7YgZCWPsTmzK2lp45mgfjL"
destination = "gunshots.wav"
!gdown --id {file_id} -O {destination}

# Check if the file was downloaded
import os
if os.path.exists(destination):
    print(f"File downloaded successfully: {destination}")
else:
    print("Download failed.")


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1IymJaN-zOk7YgZCWPsTmzK2lp45mgfjL
To: /content/gunshots.wav
100% 320k/320k [00:00<00:00, 131MB/s]
File downloaded successfully: gunshots.wav


In [5]:

display(Audio("/content/gunshots.wav"))

#  Audio Source Separation (Gunshots/Male Voice)

NOTE: `predict_spans` and `reranking_candidates` have a large impact on performance. Setting `predict_span=True` and `reranking_candidates=8` will give you better results at the cost of latency and memory. See the "Span Prediction" section below for more details

In [3]:
# Load audio file
audio_file = "/content/gunshots.wav"

# Describe the sounds you want to isolate
descriptions = ["gunshots", "male voice"]

# Same audio for each description
audios = [audio_file] * len(descriptions)

# Process and separate
batch = processor(
    audios=audios,
    descriptions=descriptions,
).to("cuda")

with torch.inference_mode():
    result = model.separate(batch, predict_spans=True, reranking_candidates=8)

# Helper: make sure tensor has shape [channels, time]
def ensure_2d(waveform: torch.Tensor) -> torch.Tensor:
    if waveform.dim() == 1:
        return waveform.unsqueeze(0)
    return waveform

# Save separated audio
sample_rate = processor.audio_sampling_rate

# result.target and result.residual are lists of tensors, one per (audio, description) pair
gunshots_target = ensure_2d(result.target[0])
male_voice_target = ensure_2d(result.target[1])

gunshots_residual = ensure_2d(result.residual[0])
male_voice_residual = ensure_2d(result.residual[1])

torchaudio.save("gunshots_target.wav", gunshots_target.cpu(), sample_rate) # The isolated sound
torchaudio.save("male_voice_target.wav", male_voice_target.cpu(), sample_rate) # The isolated sound

torchaudio.save("gunshots_residual.wav", gunshots_residual.cpu(), sample_rate) # Everything else
torchaudio.save("male_voice_residual.wav", male_voice_residual.cpu(), sample_rate) # Everything else


/usr/local/lib/python3.12/dist-packages/torch/backends/cudnn/__init__.py:145: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  torch._C._get_cudnn_allow_tf32(),
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:312: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


# Test Results

In [7]:
display(Audio("/content/gunshots_target.wav"))

In [8]:
display(Audio("/content/male_voice_target.wav"))